# 🔬  AI Data Concierge - Reproducible Analysis

<a href="https://colab.research.google.com/" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

---

## 📋 Query
> **What was the violent crime rate in pittsburgh from 2018 to 2022?**

## 📊 Metadata
| Property | Value |
|----------|-------|
| **Generated** | 2026-06-15 13:49:33 |
| **Data Source** | WPRDC |
| **Sources Used** | Western PA Regional Data Center (WPRDC) |
| **Notebook Version** | 1.0 (Colab Compatible) |

---

## 📖 How to Use This Notebook

This notebook reproduces the exact analysis performed by the ** AI Data Concierge**.
Follow the steps below to verify, modify, or extend the analysis.

### ✅ Quick Start
1. **Run All Cells**: Click `Runtime` → `Run all` (or press `Ctrl+F9`)
2. **Wait for Setup**: The first cells install dependencies and configure the environment

### 🔧 What You Can Do
| Action | Description |
|--------|-------------|
| **Verify** | Run all cells to confirm the original results |
| **Modify** | Change parameters (dates, locations, filters) and re-run |
| **Extend** | Add your own analysis cells below the results |
| **Export** | Download results as CSV, or save notebook to Drive |

### 📚 Notebook Structure
1. **Setup** - Install dependencies (runs once in Colab)
2. **Configuration** - Import libraries and set up API connections
3. **Data Retrieval** - Fetch data from the data source
4. **Analysis** - Process and analyze the data
5. **Results** - View the final answer and confidence scores
6. **Citations** - Reference sources for your research

---


In [ ]:
# ============================================================
# STEP 1: Environment Setup
# ============================================================
# This cell installs all required packages for Google Colab.
# If running locally, you can skip this cell if packages are installed.

# Check if running in Google Colab
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🔧 Running in Google Colab - Installing dependencies...")
    !pip install -q pandas numpy matplotlib seaborn requests
    print("✅ Dependencies installed successfully!")
else:
    print("💻 Running locally - assuming dependencies are installed")
    print("   If not, run: pip install pandas numpy matplotlib seaborn requests")


In [ ]:
# ============================================================
# STEP 2: Import Libraries
# ============================================================
# These are the core libraries used throughout this notebook.
# Each import serves a specific purpose in the data pipeline.

import requests      # For making HTTP requests to data APIs
import pandas as pd  # For data manipulation and analysis
import numpy as np   # For numerical computations
from datetime import datetime  # For timestamp handling
import json          # For JSON parsing

# Visualization libraries (with graceful fallback)
try:
    import matplotlib.pyplot as plt  # For creating plots
    import seaborn as sns            # For statistical visualizations
    VISUALIZATION_AVAILABLE = True
    plt.style.use('seaborn-v0_8-whitegrid')
    print("📊 Visualization libraries loaded successfully")
except ImportError:
    VISUALIZATION_AVAILABLE = False
    print("⚠️ Visualization libraries not available")
    print("   Install with: pip install matplotlib seaborn")

# ============================================================
# STEP 3: Configuration
# ============================================================
# Data source configuration - modify these if needed

CKAN_URL = "https://data.wprdc.org"

# Display configuration info
print(f"\n📅 Notebook generated: {datetime.now().isoformat()}")
print(f"🔗 CKAN URL: {CKAN_URL}")
print(f"📌 Timestamp: 2026-06-15T13:49:33.844052")


In [ ]:
# ============================================================
# STEP 4: Helper Functions
# ============================================================
# These utility functions handle data fetching from the CKAN API.
# You can reuse these functions for your own data exploration.

def fetch_ckan_data(resource_id: str, limit: int = 10000, filters: dict = None) -> pd.DataFrame:
    """
    Fetch data from CKAN DataStore API.

    This function handles pagination automatically and returns all records
    up to the specified limit.

    Parameters:
    -----------
    resource_id : str
        The unique identifier for the CKAN resource (dataset)
    limit : int, default=10000
        Maximum number of records to fetch
    filters : dict, optional
        Field filters to apply (e.g., {"state": "CA"})

    Returns:
    --------
    pd.DataFrame
        A DataFrame containing the fetched records

    Example:
    --------
    >>> df = fetch_ckan_data("abc123", limit=1000, filters={"year": 2023})
    >>> print(f"Loaded {len(df)} records")
    """
    print(f"📥 Fetching data from resource: {resource_id}")

    all_records = []
    offset = 0
    batch_size = min(32000, limit)

    while offset < limit:
        params = {
            "resource_id": resource_id,
            "limit": min(batch_size, limit - offset),
            "offset": offset,
        }

        if filters:
            params["filters"] = filters

        response = requests.post(
            f"{CKAN_URL}/api/3/action/datastore_search",
            json=params,
            headers={"Content-Type": "application/json"},
        )

        if response.status_code != 200:
            print(f"❌ Error: {response.status_code} - {response.text}")
            break

        result = response.json()
        if not result.get("success"):
            print(f"❌ CKAN error: {result.get('error')}")
            break

        records = result.get("result", {}).get("records", [])
        if not records:
            break

        all_records.extend(records)
        offset += len(records)

        total = result.get("result", {}).get("total", 0)
        print(f"   Progress: {len(all_records):,} / {total:,} records")

        if offset >= total:
            break

    df = pd.DataFrame(all_records)

    # Remove CKAN internal fields that aren't useful for analysis
    internal_cols = ["_id", "_full_text"]
    df = df.drop(columns=[c for c in internal_cols if c in df.columns], errors="ignore")

    print(f"✅ Loaded {len(df):,} records with {len(df.columns)} columns")
    return df


def search_ckan_packages(query: str, rows: int = 10) -> list:
    """
    Search CKAN for datasets (packages) matching a query.

    Parameters:
    -----------
    query : str
        The search query (e.g., "employment statistics")
    rows : int, default=10
        Maximum number of results to return

    Returns:
    --------
    list
        A list of matching package dictionaries

    Example:
    --------
    >>> packages = search_ckan_packages("housing prices", rows=5)
    >>> for pkg in packages:
    ...     print(pkg['title'])
    """
    print(f"🔍 Searching for: '{query}'")

    response = requests.post(
        f"{CKAN_URL}/api/3/action/package_search",
        json={"q": query, "rows": rows},
        headers={"Content-Type": "application/json"},
    )

    if response.status_code != 200:
        print(f"❌ Search failed: {response.status_code}")
        return []

    result = response.json()
    if not result.get("success"):
        print(f"❌ Search error: {result.get('error')}")
        return []

    packages = result.get("result", {}).get("results", [])
    print(f"✅ Found {len(packages)} matching datasets")
    return packages


print("✅ Helper functions loaded successfully!")
print("   - fetch_ckan_data(resource_id, limit, filters)")
print("   - search_ckan_packages(query, rows)")


# Data Retrieval via MCP

The following steps show how data was retrieved through the **U.S. Census Bureau MCP server**. Each step includes reproducible code you can run directly.

---

## Step 1: Lookup_Agency

**Source:** Fbi Crime Data


**Data retrieved:**
```
{
  "MCKEAN": [
    {
      "ori": "PA0421500",
      "counties": "MCKEAN",
      "is_nibrs": false,
      "latitude": 41.948418,
      "longitude": -78.66883,
      "state_abbr": "PA",
      "state_name": "Pennsylvania",
      "agency_name": "University of Pittsburgh: Bradford",
      "agency_type_name": "University or College",
      "nibrs_start_date": null
    }
  ],
  "CAMBRIA": [
    {
      "ori": "PA0116500",
      "counties": "CAMBRIA",
      "is_nibrs": false,
      "latitude": 40.270958,
      "longitude": -78.83097,
      "state_abbr": "PA",
      "state_name": "Pennsylvania",
      "agency_name": "University of Pittsburgh: Johnstown",
      "agency_type_name": "University or College",
      "nibrs_start_date": null
    }
  ],
  "CRAWFORD": [
    {
      "ori": "PA0201900",
   
```


In [ ]:
# Step 1: Lookup_Agency

# MCP Tool Call: lookup_agency (via fbi-crime-data)
# This data was retrieved using an MCP server tool.
# To reproduce, you can call the Census Bureau API directly or use the MCP server.

# Tool: lookup_agency
# Arguments:
# {
  "lookup_type": "by_state",
  "state": "PA",
  "name_filter": "Pittsburgh"
}

# Result:
result = """{
  "MCKEAN": [
    {
      "ori": "PA0421500",
      "counties": "MCKEAN",
      "is_nibrs": false,
      "latitude": 41.948418,
      "longitude": -78.66883,
      "state_abbr": "PA",
      "state_name": "Pennsylvania",
      "agency_name": "University of Pittsburgh: Bradford",
      "agency_type_name": "University or College",
      "nibrs_start_date": null
    }
  ],
  "CAMBRIA": [
    {
      "ori": "PA0116500",
      "counties": "CAMBRIA",
      "is_nibrs": false,
      "latitude": 40.270958,
      "longitude": -78.83097,
      "state_abbr": "PA",
      "state_name": "Pennsylvania",
      "agency_name": "University of Pittsburgh: Johnstown",
      "agency_type_name": "University or College",
      "nibrs_start_date": null
    }
  ],
  "CRAWFORD": [
    {
      "ori": "PA0201900",
      "counties": "CRAWFORD",
      "is_nibrs": false,
      "latitude": 41.629498,
      "longitude": -79.66508,
      "state_abbr": "PA",
      "state_name": "Pennsylvania",
      "agency_name": "University of Pittsburgh: Titusville",
      "agency_type_name": "University or College",
      "nibrs_start_date": null
    }
  ],
  "ALLEGHENY": [
    {
      "ori": "PA0021N00",
      "counties": "ALLEGHENY",
      "is_nibrs": false,
      "latitude": 40.46892,
      "longitude": -79.98092,
      "state_abbr": "PA",
      "state_name": "Pennsylvania",
      "agency_name": "University of Pittsburgh: Pittsburgh",
      "agency_type_name": "University or College",
      "nibrs_start_date": null
    },
    {
      "ori": "PAPPD0000",
      "counties": "ALLEGHENY",
      "is_nibrs": true,
      "latitude": 40.45084,
      "longitude": -80.022484,
      "state_abbr": "PA",
      "state_name": "Pennsylvania",
      "agency_name": "Pittsburgh Bureau of Police",
      "agency_type_name": "City",
      "nibrs_start_date": "2023-11-01"
    }
  ],
  "WESTMORELAND": [
    {
      "ori": "PA0655600",
      "counties": "WESTMORELAND",
      "is_nibrs": false,
      "latitude": 40.277306,
      "longitu"""
print(result)


## Step 2: Get_Summarized_Crime_Data

**Source:** Fbi Crime Data


**Data retrieved:**
```
{
  "offenses": {
    "rates": {
      "Pennsylvania Offenses": {
        "2018": 25.53,
        "2019": 24.82,
        "2020": 27.2,
        "2021": 33.58,
        "2022": 26.61
      },
      "United States Offenses": {
        "2018": 32.07,
        "2019": 31.75,
        "2020": 33.26,
        "2021": 30.16,
        "2022": 33.18
      },
      "Pennsylvania Clearances": {
        "2018": 14.23,
        "2019": 13.69,
        "2020": 13.31,
        "2021": 12.65,
        "2022": 11.79
      },
      "United States Clearances": {
        "2018": 13.82,
        "2019": 13.64,
        "2020": 13.67,
        "2021": 11.65,
        "2022": 12.33
      }
    },
    "actuals": {
      "Pennsylvania Offenses": {
        "2018": 39156,
        "2019": 36821,
        "2020": 33249,
        "2021
```


In [ ]:
# Step 2: Get_Summarized_Crime_Data

# MCP Tool Call: get_summarized_crime_data (via fbi-crime-data)
# This data was retrieved using an MCP server tool.
# To reproduce, you can call the Census Bureau API directly or use the MCP server.

# Tool: get_summarized_crime_data
# Arguments:
# {
  "offense": "V",
  "level": "state",
  "from_date": "01-2018",
  "to_date": "12-2022",
  "state": "PA"
}

# Result:
result = """{
  "offenses": {
    "rates": {
      "Pennsylvania Offenses": {
        "2018": 25.53,
        "2019": 24.82,
        "2020": 27.2,
        "2021": 33.58,
        "2022": 26.61
      },
      "United States Offenses": {
        "2018": 32.07,
        "2019": 31.75,
        "2020": 33.26,
        "2021": 30.16,
        "2022": 33.18
      },
      "Pennsylvania Clearances": {
        "2018": 14.23,
        "2019": 13.69,
        "2020": 13.31,
        "2021": 12.65,
        "2022": 11.79
      },
      "United States Clearances": {
        "2018": 13.82,
        "2019": 13.64,
        "2020": 13.67,
        "2021": 11.65,
        "2022": 12.33
      }
    },
    "actuals": {
      "Pennsylvania Offenses": {
        "2018": 39156,
        "2019": 36821,
        "2020": 33249,
        "2021": 22165,
        "2022": 32205
      },
      "Pennsylvania Clearances": {
        "2018": 21826,
        "2019": 20327,
        "2020": 16282,
        "2021": 8335,
        "2022": 14347
      }
    }
  },
  "populations": {
    "population": {
      "Pennsylvania": {
        "2018": 12807060,
        "2019": 12801989,
        "2020": 12783254,
        "2021": 12964004,
        "2022": 12972008
      },
      "United States": {
        "2018": 330362587,
        "2019": 331433049,
        "2020": 332726731,
        "2021": 335395039,
        "2022": 336746447
      }
    }
  },
  "cde_properties": {
    "max_data_date": {
      "UCR": "05/2026"
    },
    "last_refresh_date": {
      "UCR": "05/15/2026"
    }
  }
}"""
print(result)


## Step 3: Get_Summarized_Crime_Data

**Source:** Fbi Crime Data


**Data retrieved:**
```
{
  "offenses": {
    "rates": {
      "Pennsylvania Offenses": {
        "2018": 25.53,
        "2019": 24.82,
        "2020": 27.2,
        "2021": 33.58,
        "2022": 26.61
      },
      "United States Offenses": {
        "2018": 32.07,
        "2019": 31.75,
        "2020": 33.26,
        "2021": 30.16,
        "2022": 33.18
      },
      "Pennsylvania Clearances": {
        "2018": 14.23,
        "2019": 13.69,
        "2020": 13.31,
        "2021": 12.65,
        "2022": 11.79
      },
      "United States Clearances": {
        "2018": 13.82,
        "2019": 13.64,
        "2020": 13.67,
        "2021": 11.65,
        "2022": 12.33
      },
      "Pittsburgh Bureau of Police Offenses": {
        "2018": 48.23,
        "2019": 47.61,
        "2020": 42.7,
        "2021": 0.04,

```


In [ ]:
# Step 3: Get_Summarized_Crime_Data

# MCP Tool Call: get_summarized_crime_data (via fbi-crime-data)
# This data was retrieved using an MCP server tool.
# To reproduce, you can call the Census Bureau API directly or use the MCP server.

# Tool: get_summarized_crime_data
# Arguments:
# {
  "offense": "V",
  "level": "agency",
  "from_date": "01-2018",
  "to_date": "12-2022",
  "state": "PA",
  "ori": "PAPPD0000"
}

# Result:
result = """{
  "offenses": {
    "rates": {
      "Pennsylvania Offenses": {
        "2018": 25.53,
        "2019": 24.82,
        "2020": 27.2,
        "2021": 33.58,
        "2022": 26.61
      },
      "United States Offenses": {
        "2018": 32.07,
        "2019": 31.75,
        "2020": 33.26,
        "2021": 30.16,
        "2022": 33.18
      },
      "Pennsylvania Clearances": {
        "2018": 14.23,
        "2019": 13.69,
        "2020": 13.31,
        "2021": 12.65,
        "2022": 11.79
      },
      "United States Clearances": {
        "2018": 13.82,
        "2019": 13.64,
        "2020": 13.67,
        "2021": 11.65,
        "2022": 12.33
      },
      "Pittsburgh Bureau of Police Offenses": {
        "2018": 48.23,
        "2019": 47.61,
        "2020": 42.7,
        "2021": 0.04,
        "2022": 42.23
      },
      "Pittsburgh Bureau of Police Clearances": {
        "2018": 28.84,
        "2019": 27.77,
        "2020": 24.75,
        "2021": 0.0,
        "2022": 24.33
      }
    },
    "actuals": {
      "Pittsburgh Bureau of Police Offenses": {
        "2018": 1751,
        "2019": 1574,
        "2020": 1530,
        "2021": 1,
        "2022": 1536
      },
      "Pittsburgh Bureau of Police Clearances": {
        "2018": 1047,
        "2019": 918,
        "2020": 887,
        "2021": 0,
        "2022": 885
      }
    }
  },
  "populations": {
    "population": {
      "Pennsylvania": {
        "2018": 12807060,
        "2019": 12801989,
        "2020": 12783254,
        "2021": 12964004,
        "2022": 12972008
      },
      "United States": {
        "2018": 330362587,
        "2019": 331433049,
        "2020": 332726731,
        "2021": 335395039,
        "2022": 336746447
      },
      "Pittsburgh Bureau of Police": {
        "2018": 302544,
        "2019": 300548,
        "2020": 298608,
        "2021": 316632,
        "2022": 303137
      }
    }
  },
  "cde_properties": {
    "max_data_date": {
      "UCR": "05/2026"
    },
    "last_refresh_d"""
print(result)


# ============================================================
# 📊 RESULTS
# ============================================================

## 💡 Answer

> **Here is a comprehensive breakdown of the **violent crime rate in Pittsburgh** from 2018 to 2022, sourced from the **FBI Crime Data Explorer (UCR/SRS data)** for the **Pittsburgh Bureau of Police**:

---

## 🔴 Pittsburgh Violent Crime Rate (per 100,000 residents)

| Year | Rate (per 100K) | Actual Offenses | Population | Clearance Rate |
|------|-----------------|-----------------|------------|----------------|
| 2018 | **48.23** | 1,751 | 302,544 | 59.8% |
| 2019 | **47.61** | 1,574 | 300,548 | 58.3% |
| 2020 | **42.70** | 1,530 | 298,608 | 57.9% |
| 2021 | ⚠️ *Data gap* | 1* | 316,632 | — |
| 2022 | **42.23** | 1,536 | 303,137 | 57.6% |

> ⚠️ **2021 Note:** The Pittsburgh Bureau of Police reported only **1 offense** for 2021, which is almost certainly a **reporting gap or data submission issue** rather than an actual statistic. This is a known limitation in UCR data — agencies sometimes fail to submit full annual reports.

---

## 📊 Key Takeaways

- **Pittsburgh's violent crime rate declined ~12.5%** from 2018 (48.23) to 2022 (42.23), excluding the anomalous 2021 data.
- Pittsburgh's rate was consistently **well above the Pennsylvania state average** (25–34 per 100K) and modestly above the **U.S. national average** (30–33 per 100K).
- The **clearance rate** (crimes solved) hovered around **57–60%** in Pittsburgh — significantly higher than both state and national averages (~12–14%).
- The drop from 2018→2020 (~1,751 to ~1,530 offenses) suggests a gradual improvement before 2022 levels stabilized.

**Source:** FBI Crime Data Explorer — Pittsburgh Bureau of Police (ORI: `PAPPD0000`), Summarized (UCR/SRS) Violent Crime data.**

---

## 🎯 Confidence Assessment

The Data Concierge evaluates the reliability of its answer using multiple factors:


> ⚠️ Confidence score not available for this query.


# ============================================================
# 📐 CONFIDENCE SCORING METHODOLOGY
# ============================================================

## How We Calculate Confidence

The Data Concierge uses a **weighted composite score** to assess the reliability
of each answer. The final confidence score is a weighted average of five independent
factors, each measuring a different aspect of answer quality.

### Scoring Formula

```
Final Score = (0.25 × Query Interpretation)
            + (0.25 × Source Authority)
            + (0.20 × Retrieval Match)
            + (0.15 × Data Recency)
            + (0.15 × Computation Reliability)
```

### Factor Descriptions

| Factor | Weight | What It Measures | How It's Calculated |
|--------|--------|------------------|---------------------|
| **Query Interpretation** | 25% | How well the system understood the query | Entity extraction confidence × intent classification confidence |
| **Source Authority** | 25% | Trustworthiness of the data source | Pre-assigned per source (BLS/Census: 0.95, Data Commons: 0.90, CKAN: 0.85) |
| **Retrieval Match** | 20% | How well the retrieved data matches the query | Retrieval score, boosted by observation count (up to 5 observations) |
| **Data Recency** | 15% | How fresh the data is | 1.0 if within expected update cycle, decays to 0.4 floor for older data |
| **Computation Reliability** | 15% | Accuracy of the computation method | By type: direct lookup 1.0, trend analysis 0.85, statistical inference 0.70 |

### Confidence Levels

| Level | Score Range | Interpretation |
|-------|-------------|----------------|
| 🟢 **HIGH** | ≥ 85% | Results are reliable and well-supported by authoritative data |
| 🟡 **MEDIUM** | 50% – 84% | Results are reasonable but may benefit from verification |
| 🔴 **LOW** | 25% – 49% | Results should be treated with caution; data may be incomplete |
| ⚫ **VERY LOW** | < 25% | Insufficient data; consider alternative sources or queries |

### Source Authority Ratings

| Data Source | Authority Score | Rationale |
|-------------|----------------|-----------|
| Bureau of Labor Statistics (BLS) | 0.95 | Official federal statistics, rigorous methodology |
| U.S. Census Bureau | 0.95 | Comprehensive national data collection |
| Bureau of Economic Analysis (BEA) | 0.95 | Official GDP and economic accounts |
| FRED (Federal Reserve) | 0.95 | Curated economic data from the Fed |
| Google Data Commons | 0.90 | Aggregated from authoritative sources |
| WPRDC (Pittsburgh) | 0.88 | Curated regional open data portal |
| Generic CKAN Portals | 0.85 | Quality varies by portal and dataset |

### Data Recency Decay

The recency score decays based on how old the data is relative to its expected
update frequency:

- **Within 1× update cycle**: 1.0 (fully current)
- **Within 2× update cycle**: 0.8
- **Within 4× update cycle**: 0.6
- **Older than 4× update cycle**: 0.4 (floor)

### Escalation Policy

When the final confidence score falls **below 50%** after **2 retrieval attempts**,
the system flags the query for human review rather than providing a potentially
unreliable answer.

---


# ============================================================
# 📚 CITATIONS & REFERENCES
# ============================================================

## 📖 Data Sources

## Data Sources and Citations

**[1]** Western PA Regional Data Center (WPRDC)
- Dataset: Open Data Portal
- URL: [https://data.wprdc.org](https://data.wprdc.org)
- Accessed: 2026-06-15

---

## 🔄 Reproducibility Guide

This notebook was automatically generated by the ** AI Data Concierge**.
Follow these steps to reproduce or extend the analysis:

### Prerequisites

```bash
pip install pandas numpy requests matplotlib seaborn
```

### Running the Notebook

| Step | Action | Notes |
|------|--------|-------|
| 1 | **Open in Colab** | Click the "Open in Colab" badge at the top |
| 2 | **Run All Cells** | `Runtime` → `Run all` or `Ctrl+F9` |
| 3 | **Wait for completion** | Dependencies install automatically in Colab |
| 4 | **Review results** | Scroll down to see the analysis results |

### ⚠️ Important Notes

- **Data freshness**: Results may differ if data sources have been updated since generation
- **API limits**: Some data sources have rate limits; wait if you encounter errors
- **Modifications**: Feel free to modify parameters and re-run cells to explore further

### 📅 Generation Info

- **Generated**: 2026-06-15 13:49:33
- **Query**: What was the violent crime rate in pittsburgh from 2018 to 2022?
- **Data Source**: WPRDC

---

*Generated by  AI Data Concierge v0.1.0*
